# Phase 1 — Steering

The causal-intervention battery (steering, patching, noising, swap — Experiments 1-4 of the
original `vconf` pipeline) applied to three Benzon self-report/ground-truth pairs instead of
`confidence`/`correctness`:

1. **`benzon:synonyms` x `nuance_defined` x `nonbinary logit`** — the calibrated winner for the
   `nuance` construct on the synonyms dataset (`notebooks_benzon/phase_0_qwen/3_synonyms.ipynb`).
2. **List elicitation x `variety`** — `benzon_data.load_list_elicitation_items`, self-report
   `sentiment.VARIETY`, ground truth `activations.list_embedding_variety`.
3. **20 Questions x `impurity`** — `twenty_questions`, self-report `sentiment.IMPURITY`, ground
   truth `metrics.gini_impurity`.

This notebook does **steering** only (§4 of the original pipeline, `vconf/exp1_steering.py`) —
patching/noising/swap follow separately.

**Bug fixed to make this possible**: `exp1_steering.run_steering` computed
`intervention_metrics`' `confidence`-labeled columns using `interventions.numeric_midpoints`,
which returns `None` for every categorical prompt and silently falls back to
`metrics.MIDPOINTS` — a 10-class array hardcoded to `sentiment.CONFIDENCE`. That's only correct
by coincidence when `cfg.sentiment is CONFIDENCE`; for a 4-class sentiment like
`NUANCE_DEFINED`/`VARIETY`/`IMPURITY`, indexing `MIDPOINTS[0..3]` silently returns
*confidence's* midpoints instead of an `IndexError`. Fixed to derive midpoints from
`cfg.sentiment.class_midpoint` directly whenever the prompt is categorical. `logit_diff_change`/
`token_changed`/`clean_logit_diff`/`intervened_logit_diff` were never affected (they never read
midpoints) — only the `*_confidence` columns were silently wrong before this fix.

Runs under the default **reduced** profile (Qwen 2.5 7B) — see `nb.describe(cfg)`'s banner
below.


In [1]:
import copy
import json as jsonlib
import os
import pathlib
import random
import sys
from dataclasses import replace

import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

HERE = pathlib.Path.cwd()
with open(HERE / "config.json") as f:
    MODEL_CONFIG = jsonlib.load(f)
MODEL_NAME = MODEL_CONFIG["model"]
os.environ["VCONF_MODEL"] = MODEL_NAME
CACHE_DIR = HERE / "cache" / MODEL_NAME
OUT_DIR = HERE / "out" / MODEL_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

from vconf import activations
from vconf import exp1_steering as S1
from vconf import ground_truth as GT
from vconf import interventions as IV
from vconf import metrics as M
from vconf import notebook as nb
from vconf import data as datamod
from vconf import pipeline
from vconf import results as R
from vconf import twenty_questions as TQ
from vconf.prompts import LIST_ELICITATION_TEMPLATE, parse_list_items
from vconf.sentiment import CONFIDENCE, NUANCE_DEFINED, VARIETY, IMPURITY, NATURAL_COMMITMENT

base_cfg = nb.run_config("gemma-categorical", name="phase1-steering")
print(nb.describe(base_cfg))
# Gemma 3 27B in bf16 (~54 GB of weights alone) does not fit on one 48 GB GPU the way
# Qwen 7B does -- shard it across whatever's visible instead of the single-device default.
device_map = MODEL_CONFIG.get("device_map") or ("auto" if nb.profile() == "paper" else None)
loaded = nb.open_model(base_cfg, device_map=device_map)


profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : confidence  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Shared steering runner

One function, reused for all three targets: given already-scored trials (`t.confidence`/
`t.class_index` populated by `pipeline.run_phase1`) and their rendered Phase-1 prompts, collect
PANL/PANL+1/CC/FCC activations, build the high-vs-low steering vector at each (layer, position),
pick a balanced test set, and run `exp1_steering.run_steering` across the full grid. `vector_n`
is scaled down from the paper's 25 for every Benzon target here — none of these datasets are
TriviaQA-scale, so demanding 25 clean high + 25 clean low trials the way §4.2 does would leave
some targets short of data; each target picks `vector_n` explicitly, sized to what it actually
has.

In [2]:
from collections import Counter


def cached_activation_store(loaded, cfg, rendered, layers, positions, trials, label):
    """`nb.activation_store`'s own caching logic (digest the trial qids, skip recollection on
    a hit), pointed at this notebook's own `CACHE_DIR/activations` instead of the project-wide
    `ACTIVATIONS_DIR` -- every target/grid-cell's activation collection (the single most
    expensive step per steering run) survives a notebook rerun without touching the cache the
    rest of the project shares.
    """
    import hashlib

    digest = hashlib.sha1("|".join(t.qid for t in trials).encode()).hexdigest()[:10]
    name = f"{label.replace('/', '-')}-n{len(rendered)}-{digest}-l{len(layers)}-p{len(positions)}"
    path = activations.activation_path(name, directory=CACHE_DIR / "activations")
    if path.exists():
        store = activations.ActivationStore.load(path)
        if store.layers == tuple(layers) and store.positions == tuple(positions):
            return store
    store = activations.collect_activations(
        loaded, rendered, layers, positions,
        trial_ids=[t.qid for t in trials], batch_size=cfg.batch_size,
    )
    store.save(path)
    return store


def run_target_steering(
    loaded, cfg, trials, rendered, *, vector_n, test_n, require_correct, label,
):
    """Steering vectors from the whole trial set, tested on a balanced subset.

    Returns ``(frame, store, vectors)``, or ``(None, None, None)`` if the self-report turned
    out to have no variance at all to steer on (a real, reportable outcome for a small
    exploratory Benzon target, not something to force a vector out of).
    """
    layers = cfg.layers
    positions = cfg.positions
    class_counts = Counter(t.class_index for t in trials if t.class_index is not None)
    dist = {cfg.sentiment.classes[i]: n for i, n in sorted(class_counts.items())}
    print(f"[{label}] {len(trials)} trials, layers={layers}, positions={positions}")
    print(f"[{label}] self-report class distribution: {dist}")
    if len(class_counts) < 2:
        only = cfg.sentiment.classes[next(iter(class_counts))]
        print(f"[{label}] SKIPPED — self-report saturated to a single class ({only!r}); "
              f"no variance exists to build a steering vector from")
        return None, None, None

    store = cached_activation_store(loaded, cfg, rendered, layers, positions, trials, label)
    high_idx, low_idx = S1.select_vector_trials(trials, n=vector_n, require_correct=require_correct)
    print(f"[{label}] vector trials: {len(high_idx)} high / {len(low_idx)} low "
          f"(confidence range high={trials[high_idx[0]].confidence:.2f}..{trials[high_idx[-1]].confidence:.2f}, "
          f"low={trials[low_idx[0]].confidence:.2f}..{trials[low_idx[-1]].confidence:.2f})")
    vectors = S1.build_steering_vectors(store, high_idx, low_idx, scale_fraction=cfg.steering_scale_fraction)

    # Rank-based test split (top/bottom test_n//2 by raw confidence) instead of
    # select_test_trials' sentiment.high_band/low_band (the *named extreme* classes) --
    # those bands are badly underpopulated on these small Benzon self-report distributions.
    # On twenty_questions/impurity specifically, the band-based selection gave only 9 test
    # trials total, 8 of them "high" and just 1 "low" -- an almost-single-trial "low" result
    # that isn't a reliable read on anything. A rank-based split keeps the same balanced,
    # test_n-sized shape every other intervention in this pair of notebooks already uses
    # (exp2_patching's test pool, exp4_swap's recipient pools).
    order = np.argsort([t.confidence for t in trials])
    half = max(1, test_n // 2)
    test_idx = np.unique(np.concatenate([order[-half:], order[:half]]))
    test_trials, test_rendered = nb.subset(trials, rendered, test_idx)
    test_confidences = [t.confidence for t in test_trials]
    print(f"[{label}] {len(test_idx)} test trials (rank-based, confidence "
          f"{min(test_confidences):.2f}..{max(test_confidences):.2f})")

    frame = S1.run_steering(loaded, test_rendered, test_trials, vectors, cfg=cfg)
    frame["target"] = label
    return frame, store, vectors


def peak_summary(frame, label):
    """One row per (position, direction): the layer with the largest |confidence_change|."""
    if frame is None:
        return pd.DataFrame([{
            "target": label, "position": None, "direction": None, "peak_layer": None,
            "confidence_change": None, "logit_diff_change": None, "token_changed_rate": None,
        }]).iloc[0:0]
    summ = R.summarize(frame, by=("position", "direction", "layer"))
    rows = []
    for (position, direction), group in summ.groupby(["position", "direction"]):
        peak = group.loc[group["confidence_change_mean"].abs().idxmax()]
        rows.append({
            "target": label, "position": position, "direction": direction,
            "peak_layer": int(peak["layer"]),
            "confidence_change": round(float(peak["confidence_change_mean"]), 4),
            "logit_diff_change": round(float(peak["logit_diff_change_mean"]), 4),
            "token_changed_rate": round(float(peak["token_changed_mean"]), 3),
        })
    return pd.DataFrame(rows)


## Target A — `benzon:synonyms` x `nuance_defined`

120 (name1, name2, template) items. Ground truth for *which trials get steered* here is just
`nuance_defined`'s own self-report value — no correctness filter (`require_correct=False`):
"nuance" is about how multi-sided the answer reads, not about being right, and yes/no synonym
questions don't have a confidence-style notion of "the model was more sure when correct" the
way the original experiment's design assumed.

**Phase 0 stays `CONFIDENCE`, not `nuance_defined`** — same convention every calibration
notebook uses (`pipeline.run_multi_sentiment`'s `phase0_cfg = replace(base_cfg,
sentiment=CONFIDENCE)`): `build_phase0_prompt` bakes its `sentiment` argument's own instruction
block into the *answer-generation* prompt, so generating the Phase-0 answer under
`nuance_defined` framing would prime every answer toward a similarly-hedged style before
`nuance_defined` is ever asked about it as a Phase-1 follow-up — entangling the self-report with
its own priming rather than testing it against a neutral answer. `nuance_defined` only enters at
Phase 1.

In [3]:
cfg_syn = nb.run_config(
    "gemma-categorical", dataset="benzon:synonyms", sentiment=NUANCE_DEFINED,
    ground_truth=GT.SynonymAnswerKey(), name="phase1-steering-synonyms",
)
print(nb.describe(cfg_syn))

items_syn = datamod.load_dataset_items(cfg_syn.dataset)
phase0_cfg_syn = replace(cfg_syn, sentiment=CONFIDENCE)
raw_syn = pipeline.run_phase0(loaded, items_syn, phase0_cfg_syn)
kept_syn, _ = pipeline.filter_positions_isolable(loaded, raw_syn, cfg_syn)
kept_syn = [t for t in kept_syn if t.valid]
rendered_kept_syn = pipeline.run_phase1(loaded, kept_syn, cfg_syn)
paired_syn = [(t, r) for t, r in zip(kept_syn, rendered_kept_syn) if t.class_index is not None]
trials_syn = [p[0] for p in paired_syn]
rendered_syn = [p[1] for p in paired_syn]
print(f"{len(trials_syn)}/{len(items_syn)} usable trials")


profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : nuance  (ground truth: synonym_answer_key)
prompt / dataset : categorical / benzon:synonyms
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


120/120 usable trials


In [4]:
frame_syn, store_syn, vectors_syn = run_target_steering(
    loaded, cfg_syn, trials_syn, rendered_syn,
    vector_n=20, test_n=24, require_correct=False, label="synonyms/nuance_defined",
)
summary_syn = peak_summary(frame_syn, "synonyms/nuance_defined")
display(summary_syn)


[synonyms/nuance_defined] 120 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[synonyms/nuance_defined] self-report class distribution: {'Flat': 104, 'Somewhat nuanced': 15, 'Nuanced': 1}


[synonyms/nuance_defined] vector trials: 20 high / 20 low (confidence range high=0.12..0.62, low=0.12..0.12)
[synonyms/nuance_defined] 24 test trials (rank-based, confidence 0.12..0.62)


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate
0,synonyms/nuance_defined,CC,high,16,-0.0217,0.1558,0.083
1,synonyms/nuance_defined,CC,low,22,-0.0650,1.0569,0.250
2,synonyms/nuance_defined,FCC,high,0,0.0000,-0.0043,0.000
3,synonyms/nuance_defined,FCC,low,16,-0.0054,0.0165,0.021
4,synonyms/nuance_defined,PANL,high,5,-0.0054,-0.0352,0.021
5,synonyms/nuance_defined,PANL,low,0,-0.0108,0.0273,0.042
6,synonyms/nuance_defined,PANL+1,high,0,0.0046,0.0378,0.062
7,synonyms/nuance_defined,PANL+1,low,5,-0.0054,-0.0339,0.021


## Target B — 20 Questions x `impurity`

`notebooks_benzon/phase_0_qwen/5_twenty_questions.ipynb`'s own game loop
(`twenty_questions.py`), scaled down from its 10-games-per-category/10-turn design (1000 trials)
to keep this run's wall-clock time reasonable — each turn is two sequential generations (a
question, then a batched Yes/No partition over the remaining keywords), so game-play itself, not
the steering sweep, is the expensive part here. Every category is still represented; only the
per-category game count drops. Checkpointed to disk turn-by-turn so an interruption doesn't lose
already-played games.

In [5]:
cfg_tq = nb.run_config("gemma-categorical", sentiment=IMPURITY, name="phase1-steering-twentyq")
print(nb.describe(cfg_tq))

N_GAMES_PER_CATEGORY = 1 if nb.profile() == "paper" else 2  # paper-profile Gemma is far slower per generate() call
N_TURNS = 10
UNIVERSE_SIZE = 100
CHECKPOINT_PATH = CACHE_DIR / "twentyq-games.json"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


def generate_text(loaded, cfg, prompt, max_new_tokens):
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    if cfg.use_chat_template:
        text = loaded.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = messages[-1]["content"]
    enc = loaded.tokenizer([text], return_tensors="pt", add_special_tokens=not cfg.use_chat_template).to(loaded.device)
    out = loaded.model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
        top_k=None, repetition_penalty=1.0, pad_token_id=loaded.tokenizer.pad_token_id,
    )
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return loaded.tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


def play_session(loaded, cfg, keywords, secret, n_turns):
    qa_history = []
    remaining = keywords
    records = []
    for i in range(n_turns):
        messages = TQ.build_conversation_for_next_question(keywords, qa_history)
        question = generate_text(loaded, cfg, messages, max_new_tokens=40)
        partition_text = generate_text(
            loaded, cfg, TQ.build_partition_prompt(question, remaining), max_new_tokens=600
        )
        yes_set, no_set = TQ.parse_partition(partition_text, remaining)
        answer = "Yes" if secret in yes_set else "No"
        remaining_after = yes_set if secret in yes_set else no_set
        records.append({
            "secret": secret, "turn": i, "qa_history_before": list(qa_history),
            "question": question, "answer": answer, "n_yes": len(yes_set), "n_no": len(no_set),
            "gini": M.gini_impurity(len(yes_set), len(no_set)),
        })
        qa_history.append((question, answer))
        remaining = remaining_after
    return qa_history, records


random.seed(0)
expected_turns = len(TQ.CATEGORIES) * N_GAMES_PER_CATEGORY * N_TURNS
if CHECKPOINT_PATH.exists():
    all_records = jsonlib.loads(CHECKPOINT_PATH.read_text())
else:
    all_records = []

if len(all_records) == expected_turns:
    print(f"[twentyq] loaded {len(all_records)} turns from checkpoint {CHECKPOINT_PATH} "
          f"(delete it to replay from scratch)")
else:
    all_records = []
    games = []
    for category in TQ.CATEGORIES:
        for game_in_category in range(N_GAMES_PER_CATEGORY):
            seed = len(games)
            game_keywords = TQ.sample_natural_keywords(n=UNIVERSE_SIZE, seed=seed, category=category)
            secret = random.choice(game_keywords)
            qa_history, records = play_session(loaded, cfg_tq, game_keywords, secret, N_TURNS)
            game_idx = len(games)
            for record in records:
                record["category"] = category
                record["game"] = game_idx
                record["keywords"] = list(game_keywords)
            games.append({"category": category, "game": game_idx, "keywords": list(game_keywords)})
            all_records.extend(records)
            CHECKPOINT_PATH.write_text(jsonlib.dumps(all_records, indent=1))
            print(f"[twentyq] played game {game_idx + 1}/{len(TQ.CATEGORIES) * N_GAMES_PER_CATEGORY} "
                  f"({category}) -> checkpointed {len(all_records)} turns")

turns_df = pd.DataFrame(all_records)
print(f"{len(all_records)} turns total")


profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : impurity  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.
[twentyq] loaded 200 turns from checkpoint /home/stud_homes/s7846062/fatass/home/thesis/experiment/code_morph/notebooks_benzon/phase_1/cache/qwen/twentyq-games.json (delete it to replay from scratch)
200 turns total


In [6]:
impurity_trials = []
for record in all_records:
    state = TQ.transcript_summary(tuple(record["keywords"]), record["qa_history_before"])
    qid = f"{record['category']}_game{record['game']}_turn{record['turn']}"
    impurity_trials.append(pipeline.Trial(qid=qid, question=state, answer=record["question"]))

impurity_rendered = pipeline.run_phase1(loaded, impurity_trials, cfg_tq)
turns_df["self_report_value"] = [t.confidence for t in impurity_trials]
turns_df["self_report"] = [IMPURITY.classes[t.class_index] for t in impurity_trials]
print(f"{len(impurity_trials)} impurity trials scored")
display(turns_df[["category", "game", "turn", "gini", "self_report", "self_report_value"]].head(10))


200 impurity trials scored


,category,game,turn,gini,self_report,self_report_value
0,animal.n.01,0,0,0.471200,Balanced,0.62
1,animal.n.01,0,1,0.312175,Balanced,0.62
2,animal.n.01,0,2,0.496800,Balanced,0.62
3,animal.n.01,0,3,0.476371,Balanced,0.62
4,animal.n.01,0,4,0.489796,Balanced,0.62
5,animal.n.01,0,5,0.218750,Skewed,0.38
6,animal.n.01,0,6,0.408163,Skewed,0.38
7,animal.n.01,0,7,0.000000,Skewed,0.38
8,animal.n.01,0,8,0.000000,Skewed,0.38
9,animal.n.01,0,9,0.320000,Skewed,0.38


In [7]:
frame_tq, store_tq, vectors_tq = run_target_steering(
    loaded, cfg_tq, impurity_trials, impurity_rendered,
    vector_n=25, test_n=24, require_correct=False, label="twenty_questions/impurity",
)
summary_tq = peak_summary(frame_tq, "twenty_questions/impurity")
display(summary_tq)


[twenty_questions/impurity] 200 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[twenty_questions/impurity] self-report class distribution: {'Lopsided': 1, 'Skewed': 127, 'Balanced': 64, 'Even': 8}
[twenty_questions/impurity] vector trials: 25 high / 25 low (confidence range high=0.62..0.88, low=0.12..0.38)


[twenty_questions/impurity] 24 test trials (rank-based, confidence 0.12..0.88)


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate
0,twenty_questions/impurity,CC,high,22,0.0150,0.1089,0.062
1,twenty_questions/impurity,CC,low,22,-0.0771,-0.3568,0.208
2,twenty_questions/impurity,FCC,high,0,-0.0054,0.0078,0.021
3,twenty_questions/impurity,FCC,low,0,0.0000,0.0686,0.000
4,twenty_questions/impurity,PANL,high,0,-0.0054,-0.0069,0.021
5,twenty_questions/impurity,PANL,low,5,-0.0108,-0.0556,0.042
6,twenty_questions/impurity,PANL+1,high,0,-0.0054,-0.0347,0.021
7,twenty_questions/impurity,PANL+1,low,0,-0.0108,-0.0512,0.042


## Target C — List elicitation x `variety`

`notebooks_benzon/phase_0_qwen/4_list_elicitation.ipynb` ran this construct on exactly 5 trials
(one generated list per category) — nowhere near the ~2x`vector_n` clean high/low trials §4.2's
steering-vector methodology needs. Scaled up here by sampling `K_PER_CATEGORY` independent lists
per category (`do_sample=True`, temperature > 0) instead of one greedy generation each, keeping
the exact same construct (`sentiment.VARIETY`, `activations.list_embedding_variety`) — just more
draws of it, the same way the reduced profile scales *how much* runs, not *what* runs.

In [8]:
cfg_le = nb.run_config(
    "gemma-categorical", dataset="benzon:list_elicitation", sentiment=VARIETY,
    name="phase1-steering-list-elicitation",
)
print(nb.describe(cfg_le))

import torch

K_PER_CATEGORY = 6 if nb.profile() == "paper" else 10  # paper-profile Gemma is far slower per generate() call
items_le = datamod.load_dataset_items(cfg_le.dataset)


def generate_list_sampled(loaded, cfg, prompt_text, seed, max_new_tokens=400):
    if cfg.use_chat_template:
        text = loaded.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}], tokenize=False, add_generation_prompt=True
        )
    else:
        text = prompt_text
    enc = loaded.tokenizer([text], return_tensors="pt", add_special_tokens=not cfg.use_chat_template).to(loaded.device)
    torch.manual_seed(seed)  # generate() takes no per-call generator kwarg; seed the global RNG instead
    out = loaded.model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.9, top_p=0.95,
        pad_token_id=loaded.tokenizer.pad_token_id,
    )
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return loaded.tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


list_trials = []
for item in items_le:
    for k in range(K_PER_CATEGORY):
        generated = generate_list_sampled(loaded, cfg_le, item.question, seed=hash((item.qid, k)) % (2**31))
        list_trials.append(pipeline.Trial(qid=f"{item.qid}_s{k}", question=item.question, answer=generated))
    print(f"[list_elicitation] {item.meta['category']}: {K_PER_CATEGORY} samples generated "
          f"(last one had {len(parse_list_items(list_trials[-1].answer))} parsed items)")

print(f"{len(list_trials)} generated lists")


profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : variety  (ground truth: correctness)
prompt / dataset : categorical / benzon:list_elicitation
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


[list_elicitation] animals: 10 samples generated (last one had 20 parsed items)


[list_elicitation] plants: 10 samples generated (last one had 20 parsed items)


[list_elicitation] minerals: 10 samples generated (last one had 20 parsed items)


[list_elicitation] concrete objects: 10 samples generated (last one had 20 parsed items)


[list_elicitation] abstract concepts: 10 samples generated (last one had 20 parsed items)
50 generated lists


In [9]:
list_rendered = pipeline.run_phase1(loaded, list_trials, cfg_le)
# no correctness notion for "variety" (t.correct already defaults to None); steering ranks on
# self-report value alone (require_correct=False below)

variety_values = [t.confidence for t in list_trials]
print(f"variety self-report range: {min(variety_values):.2f}..{max(variety_values):.2f}, "
      f"distinct classes hit: {sorted(set(t.class_index for t in list_trials))}")


variety self-report range: 0.88..0.88, distinct classes hit: [3]


In [10]:
frame_le, store_le, vectors_le = run_target_steering(
    loaded, cfg_le, list_trials, list_rendered,
    vector_n=10, test_n=16, require_correct=False, label="list_elicitation/variety",
)
summary_le = peak_summary(frame_le, "list_elicitation/variety")
display(summary_le)


[list_elicitation/variety] 50 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[list_elicitation/variety] self-report class distribution: {'Highly varied': 50}
[list_elicitation/variety] SKIPPED — self-report saturated to a single class ('Highly varied'); no variance exists to build a steering vector from


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate


## Target D — TriviaQA x `confidence` (Qwen; the paper's own baseline)

The original confidence/correctness pair (§2.5.1) this whole reproduction is built around, run
here as a same-pipeline baseline alongside the three Benzon targets above — `nb.run_config`'s
`qwen-categorical` preset (§12.2's own axis) leaves every field but `model_key` at its default
(`dataset="triviaqa"`, `sentiment=CONFIDENCE`, `ground_truth=None` -> `AliasCorrectness`). Under
the default **reduced** profile `nb.run_config` already substitutes Qwen for *any* preset
(`gemma-categorical` included — that's why `loaded` above is already Qwen 2.5 7B, not Gemma), so
this target reuses the same loaded model rather than opening a second one; naming the preset
explicitly only matters if this notebook is ever run under `VCONF_PROFILE=paper`, where that
substitution doesn't happen and `loaded` would stay Gemma — this target is reduced-profile-only
for that reason, the same caveat `patching_noising_swap.ipynb` already notes.

Unlike the three Benzon self-reports, TriviaQA/confidence has a real correctness notion
(`nb.graded`, gold-alias matching or GPT-4o-mini when available), so `require_correct=True` is
used for vector selection here (§4.2) — the Benzon targets all use `require_correct=False`
because none of them has a meaningful right/wrong answer.

In [11]:
cfg_trivia = nb.run_config("qwen-categorical", name="phase1-steering-triviaqa")
print(nb.describe(cfg_trivia))

# 300, not cfg_trivia.trial_counts["steering"] (24 under the reduced profile -- too few correct
# trials for a 25-high/25-low vector, §4.2), and not TRIAL_COUNTS["qwen"]["steering"] = 150
# either -- 300 matches an existing on-disk cache (nb.build_trials keys its cache purely on
# model/prompt/dataset/n, not on cfg.name), so this reuses already-computed Phase-0/Phase-1
# generations instead of spending fresh GPU time on ones we already have.
TRIVIA_N = 300
trivia_trials, trivia_rendered = nb.build_trials(loaded, cfg_trivia, target_n=TRIVIA_N)
trivia_trials = nb.graded(trivia_trials)
print(f"{len(trivia_trials)} trials | {nb.grader_note()}")

profile          : reduced
model            : Qwen/Qwen2.5-7B-Instruct (28 layers)
sentiment        : confidence  (ground truth: correctness)
prompt / dataset : categorical / triviaqa
layer sweep      : (0, 5, 11, 16, 22, 27)
trial counts     : {'steering': 24, 'patching': 24, 'noising': 32, 'swap': 24, 'attention': 24}
activation set   : 300   calibration set: 40
chat template    : True   attention impl: None
NOTE             : reduced profile — procedures, prompts, positions and
                   metrics follow the manual exactly, but the model and the
                   sample sizes are smaller than the paper's, so the numbers
                   here are not expected to match its reported values.


300 trials | OPENAI_API_KEY is not set, so correctness labels come from the documented fallback (normalised alias matching), not the manual's gpt-4o-mini grader (§2.3.1) — accuracy/ECE/AUROC here are approximate for that reason


In [12]:
frame_trivia, store_trivia, vectors_trivia = run_target_steering(
    loaded, cfg_trivia, trivia_trials, trivia_rendered,
    vector_n=25, test_n=24, require_correct=True, label="triviaqa/confidence",
)
summary_trivia = peak_summary(frame_trivia, "triviaqa/confidence")
display(summary_trivia)

[triviaqa/confidence] 300 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[triviaqa/confidence] self-report class distribution: {'No chance': 3, 'Really unlikely': 12, 'Unlikely': 29, 'Likely': 75, 'Highly likely': 127, 'Almost certain': 54}


[triviaqa/confidence] vector trials: 25 high / 25 low (confidence range high=0.95..0.95, low=0.35..0.65)
[triviaqa/confidence] 24 test trials (rank-based, confidence 0.05..0.95)


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate
0,triviaqa/confidence,CC,high,22,0.0896,0.9337,0.146
1,triviaqa/confidence,CC,low,22,-0.0458,-1.4248,0.479
2,triviaqa/confidence,FCC,high,0,0.0083,0.0441,0.083
3,triviaqa/confidence,FCC,low,5,0.0083,0.0172,0.083
4,triviaqa/confidence,PANL,high,16,0.0271,-0.2487,0.104
5,triviaqa/confidence,PANL,low,16,-0.0187,-0.0823,0.271
6,triviaqa/confidence,PANL+1,high,5,0.0042,-0.0045,0.083
7,triviaqa/confidence,PANL+1,low,11,-0.0062,-0.0255,0.062


## Combined summary

All four targets' peak layer/position deltas side by side — the same shape as
`exp1_steering.PAPER_TARGETS`, but for the three Benzon (sentiment, dataset) pairs plus the
paper's own confidence/TriviaQA baseline (Target D) run through the identical pipeline. A real
"PANL before CC" signature here would look like: PANL's peak layer is earlier than CC's, and
PANL's steering effect (`confidence_change`) is comparable in magnitude to CC's rather than
negligible next to it.

In [13]:
combined = pd.concat([summary_syn, summary_tq, summary_le, summary_trivia], ignore_index=True)
combined.to_csv(OUT_DIR / "combined_summary.csv", index=False)
display(combined.sort_values(["target", "position", "direction"]))


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate
0,synonyms/nuance_defined,CC,high,16,-0.0217,0.1558,0.083
1,synonyms/nuance_defined,CC,low,22,-0.065,1.0569,0.25
2,synonyms/nuance_defined,FCC,high,0,0.0,-0.0043,0.0
3,synonyms/nuance_defined,FCC,low,16,-0.0054,0.0165,0.021
4,synonyms/nuance_defined,PANL,high,5,-0.0054,-0.0352,0.021
5,synonyms/nuance_defined,PANL,low,0,-0.0108,0.0273,0.042
6,synonyms/nuance_defined,PANL+1,high,0,0.0046,0.0378,0.062
7,synonyms/nuance_defined,PANL+1,low,5,-0.0054,-0.0339,0.021
16,triviaqa/confidence,CC,high,22,0.0896,0.9337,0.146
17,triviaqa/confidence,CC,low,22,-0.0458,-1.4248,0.479


In [14]:
for target_name, group in combined.groupby("target"):
    panl = group[(group["position"] == "PANL")]
    cc = group[(group["position"] == "CC")]
    if len(panl) and len(cc):
        panl_layer = panl["peak_layer"].mean()
        cc_layer = cc["peak_layer"].mean()
        panl_effect = panl["confidence_change"].abs().mean()
        cc_effect = cc["confidence_change"].abs().mean()
        order = "PANL before CC" if panl_layer < cc_layer else "CC before/at PANL"
        print(f"{target_name}: PANL peak L{panl_layer:.1f} (|delta|={panl_effect:.3f})  "
              f"CC peak L{cc_layer:.1f} (|delta|={cc_effect:.3f})  -> {order}")


synonyms/nuance_defined: PANL peak L2.5 (|delta|=0.008)  CC peak L19.0 (|delta|=0.043)  -> PANL before CC
triviaqa/confidence: PANL peak L16.0 (|delta|=0.023)  CC peak L22.0 (|delta|=0.068)  -> PANL before CC
twenty_questions/impurity: PANL peak L2.5 (|delta|=0.008)  CC peak L22.0 (|delta|=0.046)  -> PANL before CC


## Grid — {TriviaQA, benzon:synonyms} x {confidence, commitment_challenge, nuance_defined} x {correctness, answer_logit, binary_entropy}

Every self-report crossed against every ground truth, on both datasets — the same "don't assume
which self-report belongs with which ground truth" design as the `phase_0_calibration/`
notebooks (`vconf/trial_log.py`'s own OM/GT vocabulary), now applied to steering instead of just
correlation. In every one of these 18 cells the self-report (`t.confidence`) still decides which
trials are "high" vs "low" for the steering vector — that part never changes — but *which ground
truth defines "answered correctly" for `require_correct=True`'s §4.2 eligibility filter* does.

- `confidence` -> `sentiment.CONFIDENCE` (Target D's own sentiment)
- `commitment_challenge` -> `sentiment.NATURAL_COMMITMENT` — "how likely will you change your
  mind if provided evidence?", named in `notebooks_benzon/phase_0_calibration/1_commitment.ipynb`
  for its *own* native ground truth (challenge-protocol logit drop), which isn't one of the three
  used here — this grid deliberately crosses it against the other three instead.
- `nuance_defined` -> `sentiment.NUANCE_DEFINED` (Target A's own sentiment)
- `correctness` -> `ground_truth.AliasCorrectness()` on TriviaQA, `ground_truth.SynonymAnswerKey()`
  on synonyms (each dataset's own real notion of "right")
- `answer_logit` -> `ground_truth.MACHINE_COMMITMENT_GROUND_TRUTH` (mean answer log-probability,
  pool-relative median split) — dataset-agnostic
- `binary_entropy` -> a fresh `IntrinsicMetricThreshold` over `metrics.answer_set_entropy`, bound
  to each dataset's own Phase-0 (`sentiment=CONFIDENCE`) config — the metric `trial_log.py`'s
  canonical `GT_KEY` actually logs under the display name "binary entropy" (key `entropy_gt`),
  *not* `2_nuance.ipynb`'s separate, later {Yes,No}-restricted `yes_no_logit_entropy` experiment
  (logged there under a different, confusingly-similar-sounding key).

Each dataset reuses the Phase-0 pool already built above — Target D's `trivia_trials` for
TriviaQA, Target A's `kept_syn` for synonyms — instead of spending a second Phase-0 pass;
`pipeline.run_phase1` is cheap to rerun per sentiment on top of an existing pool (one forward
pass per trial, no generation), and is shared across that sentiment's three ground-truth cells
too (only `t.correct` and the steering sweep differ between them).

**Known limitation, visible already in Target A's own printed diagnostic above ("3/120 usable
trials"):** `benzon:synonyms`'s short Yes/No Phase-0 answers very often fail
`filter_positions_isolable`'s PANL-isolation check (a trailing period merging with the following
newline — exactly the BPE failure mode `pipeline.py`'s own docstring warns about), so every
synonyms cell below inherits that same ~3-trial pool regardless of which sentiment or ground
truth it's testing. Most or all 9 synonyms cells are expected to come back SKIPPED (too few
trials for even a minimal vector) — that's a real property of this dataset/tokenizer pairing,
not a bug in this grid.

In [15]:
def sentiment_trials_from_pool(loaded, base_trials, base_cfg, sentiment):
    """One Phase-1 pass over a deep copy of an already Phase-0'd, isolable-filtered pool --
    `pipeline.run_multi_sentiment`'s own sharing trick, starting from a pool this notebook
    already built (Target A/D above) instead of a fresh Phase-0 pass."""
    cfg = replace(base_cfg, sentiment=sentiment)
    trial_copies = copy.deepcopy(base_trials)
    rendered = pipeline.run_phase1(loaded, trial_copies, cfg)
    return cfg, trial_copies, rendered


def cached_answer_set_entropy(loaded, dataset, phase0_cfg):
    """`metrics.answer_set_entropy`, cached to disk by (dataset, qid). Its value only depends
    on the question/Phase-0 config, not on which follow-up sentiment is under test, so every
    sentiment sharing this dataset reuses the same on-disk values instead of recomputing them
    (a real forward pass per trial) three times over -- once per sentiment in the grid below.
    """
    cache_path = CACHE_DIR / "answer_set_entropy.json"
    cache = jsonlib.loads(cache_path.read_text()) if cache_path.exists() else {}
    dataset_cache = cache.setdefault(dataset, {})

    def metric_fn(trial):
        if trial.qid not in dataset_cache:
            dataset_cache[trial.qid] = M.answer_set_entropy(trial, loaded, phase0_cfg)
            cache_path.write_text(jsonlib.dumps(cache))
        return dataset_cache[trial.qid]

    return metric_fn


def ground_truths_for(loaded, dataset, phase0_cfg):
    """The three ground-truth definitions this grid gates `require_correct` on."""
    correctness_gt = GT.SynonymAnswerKey() if dataset.startswith("benzon:") else GT.AliasCorrectness()
    return {
        "correctness": correctness_gt,
        "answer_logit": GT.MACHINE_COMMITMENT_GROUND_TRUTH,
        "binary_entropy": GT.IntrinsicMetricThreshold(
            name="entropy_gt", metric_fn=cached_answer_set_entropy(loaded, dataset, phase0_cfg)
        ),
    }


def apply_ground_truth(trials, items, ground_truth):
    """Populate `t.correct` via `ground_truth`, matching each trial to its item by qid (not a
    blind positional zip -- `trials` here is a filtered *subset* of `items` in general).
    `nb.graded`'s own reconstructed QuestionItems lose `item.meta` entirely, so a meta-based
    ground truth (`SynonymAnswerKey`) needs the dataset's real items passed in, not that
    reconstruction; `items=trials` is fine for a ground truth that never reads item fields
    (`AliasCorrectness`, `IntrinsicMetricThreshold`)."""
    by_qid = {item.qid: item for item in items}
    matched_items = [by_qid[t.qid] for t in trials]
    labels = ground_truth.labels(matched_items, trials)
    for trial, label in zip(trials, labels):
        trial.correct = bool(label)
    return trials

In [16]:
GRID_DATASETS = ("triviaqa", "synonyms")
GRID_SENTIMENTS = {
    "confidence": CONFIDENCE, "commitment_challenge": NATURAL_COMMITMENT, "nuance_defined": NUANCE_DEFINED,
}
GRID_GROUND_TRUTHS = ("correctness", "answer_logit", "binary_entropy")

DATASET_POOLS = {
    "triviaqa": {
        "pool": trivia_trials, "phase0_cfg": replace(cfg_trivia, sentiment=CONFIDENCE),
        "items": datamod.load_dataset_items("triviaqa"), "gt_dataset": "triviaqa",
    },
    "synonyms": {
        "pool": kept_syn, "phase0_cfg": replace(cfg_syn, sentiment=CONFIDENCE),
        "items": items_syn, "gt_dataset": "benzon:synonyms",
    },
}

# Full 18-cell grid; SANITY_CHECK restricts to one already-proven combo (triviaqa/confidence/
# correctness, == Target D) plus one genuinely new one (a different dataset, sentiment, AND
# ground truth all at once) so the new wiring is checked before committing to the full sweep's
# GPU time. Flip to False for the real run.
SANITY_CHECK = False
GRID_COMBOS = (
    [("triviaqa", "confidence", "correctness"), ("synonyms", "commitment_challenge", "answer_logit")]
    if SANITY_CHECK
    else [(d, s, g) for d in GRID_DATASETS for s in GRID_SENTIMENTS for g in GRID_GROUND_TRUTHS]
)
print(f"running {len(GRID_COMBOS)} of 18 grid cells (SANITY_CHECK={SANITY_CHECK})")

running 18 of 18 grid cells (SANITY_CHECK=False)


In [17]:
grid_rows = []
# One Phase-1 pass per (dataset, sentiment), reused across that sentiment's three
# ground-truth cells below -- only `t.correct` and the steering sweep differ between them.
sentiment_cache = {}
for dataset_key, sentiment_key, gt_key in GRID_COMBOS:
    spec = DATASET_POOLS[dataset_key]
    cache_key = (dataset_key, sentiment_key)
    if cache_key not in sentiment_cache:
        sentiment_cache[cache_key] = sentiment_trials_from_pool(
            loaded, spec["pool"], spec["phase0_cfg"], GRID_SENTIMENTS[sentiment_key]
        )
    cfg, trials, rendered = sentiment_cache[cache_key]

    gt = ground_truths_for(loaded, spec["gt_dataset"], spec["phase0_cfg"])[gt_key]
    trial_copies = copy.deepcopy(trials)
    apply_ground_truth(trial_copies, spec["items"], gt)

    label = f"{dataset_key}/{sentiment_key}/{gt_key}"
    n_eligible = sum(1 for t in trial_copies if t.correct)
    print(f"[{label}] {len(trial_copies)} trials, {n_eligible} eligible under require_correct=True")
    if n_eligible < 6:
        print(f"[{label}] SKIPPED -- only {n_eligible} eligible trials (need >= 6 for even a minimal vector)")
        grid_rows.append(peak_summary(None, label))
        continue

    vector_n = min(20, n_eligible // 2)
    frame, _, _ = run_target_steering(
        loaded, cfg, trial_copies, rendered, vector_n=vector_n, test_n=min(24, len(trial_copies)),
        require_correct=True, label=label,
    )
    grid_rows.append(peak_summary(frame, label))

grid_summary = pd.concat(grid_rows, ignore_index=True)
grid_summary.to_csv(OUT_DIR / "grid_summary.csv", index=False)
display(grid_summary)

[triviaqa/confidence/correctness] 300 trials, 185 eligible under require_correct=True
[triviaqa/confidence/correctness] 300 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[triviaqa/confidence/correctness] self-report class distribution: {'No chance': 3, 'Really unlikely': 13, 'Unlikely': 27, 'Likely': 75, 'Very good chance': 1, 'Highly likely': 127, 'Almost certain': 54}


[triviaqa/confidence/correctness] vector trials: 20 high / 20 low (confidence range high=0.95..0.95, low=0.15..0.65)
[triviaqa/confidence/correctness] 24 test trials (rank-based, confidence 0.05..0.95)


[triviaqa/confidence/answer_logit] 300 trials, 150 eligible under require_correct=True
[triviaqa/confidence/answer_logit] 300 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[triviaqa/confidence/answer_logit] self-report class distribution: {'No chance': 3, 'Really unlikely': 13, 'Unlikely': 27, 'Likely': 75, 'Very good chance': 1, 'Highly likely': 127, 'Almost certain': 54}


[triviaqa/confidence/answer_logit] vector trials: 20 high / 20 low (confidence range high=0.95..0.95, low=0.05..0.65)
[triviaqa/confidence/answer_logit] 24 test trials (rank-based, confidence 0.05..0.95)


[triviaqa/confidence/binary_entropy] 300 trials, 150 eligible under require_correct=True
[triviaqa/confidence/binary_entropy] 300 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[triviaqa/confidence/binary_entropy] self-report class distribution: {'No chance': 3, 'Really unlikely': 13, 'Unlikely': 27, 'Likely': 75, 'Very good chance': 1, 'Highly likely': 127, 'Almost certain': 54}


...

[synonyms/nuance_defined/answer_logit] 120 trials, 60 eligible under require_correct=True
[synonyms/nuance_defined/answer_logit] 120 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[synonyms/nuance_defined/answer_logit] self-report class distribution: {'Flat': 104, 'Somewhat nuanced': 15, 'Nuanced': 1}


[synonyms/nuance_defined/answer_logit] vector trials: 20 high / 20 low (confidence range high=0.12..0.38, low=0.12..0.12)
[synonyms/nuance_defined/answer_logit] 24 test trials (rank-based, confidence 0.12..0.62)


[synonyms/nuance_defined/binary_entropy] 120 trials, 60 eligible under require_correct=True
[synonyms/nuance_defined/binary_entropy] 120 trials, layers=(0, 5, 11, 16, 22, 27), positions=('PANL', 'PANL+1', 'CC', 'FCC')
[synonyms/nuance_defined/binary_entropy] self-report class distribution: {'Flat': 104, 'Somewhat nuanced': 15, 'Nuanced': 1}


[synonyms/nuance_defined/binary_entropy] vector trials: 20 high / 20 low (confidence range high=0.12..0.38, low=0.12..0.12)
[synonyms/nuance_defined/binary_entropy] 24 test trials (rank-based, confidence 0.12..0.62)


,target,position,direction,peak_layer,confidence_change,logit_diff_change,token_changed_rate
0,triviaqa/confidence/correctness,CC,high,22,0.0708,0.8064,0.125
1,triviaqa/confidence/correctness,CC,low,22,-0.0500,-1.3064,0.417
2,triviaqa/confidence/correctness,FCC,high,11,0.0042,-0.0211,0.042
3,triviaqa/confidence/correctness,FCC,low,22,-0.0042,-0.0161,0.042
4,triviaqa/confidence/correctness,PANL,high,16,0.0271,-0.2938,0.104
...,...,...,...,...,...,...,...
139,synonyms/nuance_defined/binary_entropy,FCC,low,11,-0.0054,-0.0295,0.021
140,synonyms/nuance_defined/binary_entropy,PANL,high,0,0.0100,-0.0456,0.042
141,synonyms/nuance_defined/binary_entropy,PANL,low,11,-0.0163,0.0434,0.062
142,synonyms/nuance_defined/binary_entropy,PANL+1,high,16,-0.0054,0.0156,0.021
